In [ ]:
!pip install pandas

# Spam Email Classifier — Naive Bayes

The goal is to classify SMS messages as spam or ham (normal messages) using Naive Bayes. No libraries like sklearn — we're building it from scratch so we actually understand what's happening under the hood.

Naive Bayes is based on Bayes' theorem. The core idea is: given a message with words $w_1, w_2, ..., w_n$, we want to figure out which is more likely — spam or ham?

$$P(\text{class} \mid \text{message}) \propto P(\text{class}) \times \prod_{i=1}^{n} P(w_i \mid \text{class})$$

We calculate this score for both spam and ham, and whichever is higher is our prediction.

It's called *naive* because it assumes each word is independent of the others — which is obviously not true in real language, but the classifier still works surprisingly well.

---
## 1. Load the data

In [ ]:
import pandas as pd

print("Loading dataset...")
df = pd.read_csv("spam.csv")
print(f"Total messages: {len(df)}")
df.head()

---
## 2. Prior probabilities

Before we even look at the words in a message, we need to know how common spam is overall. This is the **prior** — basically the base rate.

$$P(\text{spam}) = \frac{\text{number of spam messages}}{\text{total messages}}$$

If spam is rare, the model starts out skeptical and needs strong word evidence to flip to spam. Makes sense.

In [ ]:
spam_count = len(df[df["Category"] == "spam"])
ham_count  = len(df[df["Category"] == "ham"])
total      = len(df)

print(f"Spam : {spam_count}")
print(f"Ham  : {ham_count}")

prior_spam = spam_count / total
prior_ham  = ham_count  / total

print(f"\nP(spam) = {prior_spam:.4f}")
print(f"P(ham)  = {prior_ham:.4f}")

---
## 3. Text preprocessing

Raw messages are noisy. `FREE`, `Free`, and `free` should all be the same word. `win!` and `win` should be the same. And words like `the`, `is`, `a` appear equally in spam and ham, so they carry no useful signal — we drop them.

Steps:
1. lowercase everything
2. remove punctuation and numbers
3. split into words
4. remove stop words

In [ ]:
import re

STOP_WORDS = {
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "they", "them",
    "it", "its", "the", "a", "an", "and", "or", "but", "in", "on", "at",
    "to", "for", "of", "is", "was", "are", "be", "been", "have", "has",
    "do", "did", "not", "no", "so", "if", "as", "by", "this", "that",
    "with", "from", "just", "now", "up", "out", "can", "will", "get",
    "got", "go", "im", "ur", "u", "r", "da", "de"
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)      # keep only letters
    tokens = text.split()
    tokens = [w for w in tokens if w not in STOP_WORDS and len(w) > 1]
    return tokens

df['tokens'] = df['Message'].apply(preprocess)

# check what a spam message looks like before and after
print("Before:", df['Message'].iloc[2])
print("After :", df['tokens'].iloc[2])

---
## 4. Word frequencies + Laplace smoothing

We count how many times each word appears in spam messages vs ham messages. Then the likelihood of a word given a class is:

$$P(w \mid \text{spam}) = \frac{\text{count}(w \text{ in spam})}{\text{total words in spam}}$$

Problem: if a word never appeared in spam, this becomes 0. And since we multiply all the word probabilities together, one zero kills the whole calculation.

Fix: **Laplace smoothing** — just add 1 to every count so nothing is ever zero.

$$P(w \mid \text{spam}) = \frac{\text{count}(w \text{ in spam}) + 1}{\text{total spam words} + |V|}$$

where $|V|$ is the size of the vocabulary (total unique words).

In [ ]:
from collections import defaultdict

spam_word_freq = defaultdict(int)
ham_word_freq  = defaultdict(int)

for token_list in df[df['Category'] == 'spam']['tokens']:
    for word in token_list:
        spam_word_freq[word] += 1

for token_list in df[df['Category'] == 'ham']['tokens']:
    for word in token_list:
        ham_word_freq[word] += 1

vocabulary       = set(spam_word_freq.keys()) | set(ham_word_freq.keys())
vocab_size       = len(vocabulary)
total_spam_words = sum(spam_word_freq.values())
total_ham_words  = sum(ham_word_freq.values())

print(f"Vocabulary size     : {vocab_size:,} unique words")
print(f"Words in spam       : {total_spam_words:,}")
print(f"Words in ham        : {total_ham_words:,}")

print("\nTop spam words:")
for word, count in sorted(spam_word_freq.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {word}: {count}")

---
## 5. Prediction

We compute the log-probability (not raw probability) for each class. Reason: multiplying hundreds of small decimals leads to underflow — the number gets so tiny the computer rounds it to 0. Taking the log converts multiplication to addition, which is numerically stable.

$$\log P(\text{spam} \mid \text{message}) = \log P(\text{spam}) + \sum_{w} \log P(w \mid \text{spam})$$

Higher log score wins.

In [ ]:
import math

def predict(message):
    tokens = preprocess(message)

    log_prob_spam = math.log(prior_spam)
    log_prob_ham  = math.log(prior_ham)

    for word in tokens:
        log_prob_spam += math.log((spam_word_freq[word] + 1) / (total_spam_words + vocab_size))
        log_prob_ham  += math.log((ham_word_freq[word]  + 1) / (total_ham_words  + vocab_size))

    return 'spam' if log_prob_spam > log_prob_ham else 'ham'


tests = [
    "Congratulations! You have won a FREE prize. Call now to claim!",
    "Hey, are you coming to dinner tonight?",
    "URGENT: Your account has been selected to receive £1000. Text CLAIM to 84788.",
    "Can you pick up some groceries on your way home?"
]

for msg in tests:
    label = predict(msg)
    print(f"[{label}] {msg[:70]}")

---
## 6. Evaluation

Accuracy alone is misleading here since ~87% of messages are ham — a model that predicts ham every time gets 87% accuracy while catching zero spam. So we also track precision, recall and F1.

| Metric | Formula |
|--------|---------|
| Accuracy | $(TP + TN)\,/\,\text{total}$ |
| Precision | $TP\,/\,(TP + FP)$ — of predicted spam, how many were actually spam |
| Recall | $TP\,/\,(TP + FN)$ — of actual spam, how many did we catch |
| F1 | $2 \cdot \frac{P \times R}{P + R}$ — balance between precision and recall |

In [ ]:
df['predicted'] = df['Message'].apply(predict)

TP = len(df[(df['Category'] == 'spam') & (df['predicted'] == 'spam')])
TN = len(df[(df['Category'] == 'ham')  & (df['predicted'] == 'ham')])
FP = len(df[(df['Category'] == 'ham')  & (df['predicted'] == 'spam')])
FN = len(df[(df['Category'] == 'spam') & (df['predicted'] == 'ham')])

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP (spam caught)      : {TP}")
print(f"TN (ham passed)       : {TN}")
print(f"FP (ham flagged)      : {FP}")
print(f"FN (spam missed)      : {FN}")
print()
print(f"Accuracy  : {accuracy*100:.2f}%")
print(f"Precision : {precision*100:.2f}%")
print(f"Recall    : {recall*100:.2f}%")
print(f"F1 Score  : {f1*100:.2f}%")